# Project04 Hand Gesture ROI Classifier

OpenCV ?? ROI?? ? ??? ???? ????, CNN ??? ?? ??? ??? ?, ?? ?? ROI? ???? ?????? ??????.

???: `open_hand`, `fist`, `peace`, `unknown`


## 1. ?????? ?? ??


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import tensorflow as tf

from project04_hand_gesture_utils import GESTURE_CLASSES, preprocess_hand_roi, predict_gesture
from project04_realtime_utils import crop_center_roi

BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / "hand_gesture_dataset"
MODEL_PATH = BASE_DIR / "project04_hand_gesture.keras"
IMAGE_SIZE = (128, 128)
ROI_SIZE = 320
CAMERA_INDEX = 0
TRAIN_EPOCHS = 25
GESTURE_CLASSES


## 2. ??? ??

?? ??? ?? OpenCV ?? ?? ROI ??? ?????.


In [ ]:
# ??? ? ??? ?? ?????.
# %run project04_hand_gesture_collect.py


## 3. ??? ?? ??


In [ ]:
def count_images(folder: Path):
    return len(list(folder.glob("*.jpg")))

counts = {class_name: count_images(DATASET_DIR / class_name) for class_name in GESTURE_CLASSES}
counts


## 4. ??? ??? ?? ??


In [ ]:
def load_images(folder: Path):
    images = []
    for image_path in sorted(folder.glob("*.jpg")):
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            continue
        image = cv2.resize(image, IMAGE_SIZE, interpolation=cv2.INTER_AREA)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        images.append(image)
    return images

def build_dataset(balance=True):
    loaded = {class_name: load_images(DATASET_DIR / class_name) for class_name in GESTURE_CLASSES}
    counts = {class_name: len(images) for class_name, images in loaded.items()}
    if any(count == 0 for count in counts.values()):
        raise FileNotFoundError(f"? ??? ??? ???? ?????: {counts}")
    sample_count = min(counts.values()) if balance else None
    x_items, y_items = [], []
    for class_index, class_name in enumerate(GESTURE_CLASSES):
        images = loaded[class_name][:sample_count]
        for image in images:
            label = np.zeros(len(GESTURE_CLASSES), dtype=np.float32)
            label[class_index] = 1.0
            x_items.append(image)
            y_items.append(label)
    x = np.asarray(x_items, dtype=np.float32) / 255.0
    y = np.asarray(y_items, dtype=np.float32)
    return x, y, counts

def build_model():
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(128, 128, 3)),
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.04),
        tf.keras.layers.RandomZoom(0.08),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(96, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(0.35),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(len(GESTURE_CLASSES), activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


## 5. ?? ?? ? ??


In [ ]:
FORCE_TRAIN = True

if MODEL_PATH.exists() and not FORCE_TRAIN:
    model = tf.keras.models.load_model(MODEL_PATH)
    history = None
else:
    x, y, counts = build_dataset(balance=True)
    print(counts)
    model = build_model()
    history = model.fit(x, y, epochs=TRAIN_EPOCHS, batch_size=16, validation_split=0.2, verbose=1)
    model.save(MODEL_PATH)
    print(f"Model saved: {MODEL_PATH}")

model.summary()


## 6. ?? ROI ??? ??


In [ ]:
# ?? ?? ??? ????. ??? q ?? ESC ???.
# %run project04_hand_gesture_webcam.py
